# Colab GPU — reranker scoring resumable

Notebook minimale per eseguire **solo** la parte costosa di reranking su GPU Colab. Non richiede il progetto completo né gli embedding: l'input è `rerank_candidates_unique.csv`, esportato dal notebook 09 aggiornato.

Il file contiene una sola riga per coppia unica `(query_id, movieId)` appartenente all'unione dei top-30 dei `FINALIST_SYSTEMS`. Ogni reranker assegna lo score una sola volta. I checkpoint vengono salvati su Google Drive dopo piccoli chunk, quindi una disconnessione non azzera il lavoro già completato.


In [ ]:
!pip -q install "FlagEmbedding==1.4.2" "sentence-transformers>=5.4,<6" "transformers>=4.51,<5"


In [ ]:
from __future__ import annotations

import gc
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from google.colab import drive

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    raise RuntimeError("Attiva Runtime > Cambia tipo di runtime > GPU prima di proseguire.")

drive.mount("/content/drive")


In [ ]:
# Metti rerank_candidates_unique.csv in questa cartella di Drive.
DRIVE_DIR = Path("/content/drive/MyDrive/movie_rag_rerank")
DRIVE_DIR.mkdir(parents=True, exist_ok=True)

CANDIDATES_PATH = DRIVE_DIR / "rerank_candidates_unique.csv"
SCORE_DIR = DRIVE_DIR / "reranker_scores"
SCORE_DIR.mkdir(parents=True, exist_ok=True)

ID_COLUMN = "movieId"
DOCUMENT_COLUMN = "document__D_full_enriched_v2"
MAX_LENGTH = 1024
CHECKPOINT_CHUNK_SIZE = 32

if not CANDIDATES_PATH.exists():
    raise FileNotFoundError(
        f"Non trovo {CANDIDATES_PATH}. Copia qui il file esportato dal notebook 09."
    )

candidates = pd.read_csv(CANDIDATES_PATH)
required = {"query_id", "query", ID_COLUMN, DOCUMENT_COLUMN}
missing = required - set(candidates.columns)
if missing:
    raise KeyError(f"Colonne mancanti: {sorted(missing)}")
if candidates.duplicated(["query_id", ID_COLUMN]).any():
    raise ValueError("Il file contiene coppie (query_id, movieId) duplicate")

print("Coppie uniche:", len(candidates))
print("Query:", candidates["query_id"].nunique())
display(candidates.head())


In [ ]:
from sentence_transformers import CrossEncoder
from FlagEmbedding import FlagReranker


class BGEReranker:
    def __init__(self, batch_size=16, max_length=1024):
        self.batch_size = batch_size
        self.max_length = max_length
        self.model = None

    def load(self):
        if self.model is None:
            self.model = FlagReranker("BAAI/bge-reranker-v2-m3", use_fp16=True)
        return self

    def score(self, query, documents):
        self.load()
        pairs = [[str(query), str(doc)] for doc in documents]
        values = self.model.compute_score(
            pairs,
            batch_size=self.batch_size,
            max_length=self.max_length,
            normalize=True,
        )
        return np.atleast_1d(np.asarray(values, dtype=float)).reshape(-1)

    def unload(self):
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()


class STReranker:
    def __init__(self, model_name, trust_remote_code=False, batch_size=8, max_length=1024):
        self.model_name = model_name
        self.trust_remote_code = trust_remote_code
        self.batch_size = batch_size
        self.max_length = max_length
        self.model = None

    def load(self):
        if self.model is None:
            self.model = CrossEncoder(
                self.model_name,
                device="cuda",
                trust_remote_code=self.trust_remote_code,
                max_length=self.max_length,
                model_kwargs={"torch_dtype": torch.float16},
            )
        return self

    def score(self, query, documents):
        self.load()
        pairs = [(str(query), str(doc)) for doc in documents]
        values = self.model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False,
        )
        return np.asarray(values, dtype=float).reshape(-1)

    def unload(self):
        self.model = None
        gc.collect()
        torch.cuda.empty_cache()


RERANKER_FACTORIES = {
    "bge_reranker_v2_m3": lambda: BGEReranker(batch_size=16, max_length=MAX_LENGTH),
    "qwen3_reranker_0.6b": lambda: STReranker(
        "Qwen/Qwen3-Reranker-0.6B", batch_size=8, max_length=MAX_LENGTH
    ),
    "gte_multilingual_reranker": lambda: STReranker(
        "Alibaba-NLP/gte-multilingual-reranker-base",
        trust_remote_code=True,
        batch_size=8,
        max_length=MAX_LENGTH,
    ),
}


def atomic_write_csv(frame: pd.DataFrame, path: Path):
    tmp = path.with_suffix(path.suffix + ".tmp")
    frame.to_csv(tmp, index=False)
    tmp.replace(path)


def score_incremental(candidates, reranker, reranker_name, checkpoint_path, chunk_size=32):
    columns = ["query_id", "query", ID_COLUMN, "reranker", "reranker_score"]
    if checkpoint_path.exists():
        checkpoint = pd.read_csv(checkpoint_path)
    else:
        checkpoint = pd.DataFrame(columns=columns)

    completed = set(zip(
        checkpoint.get("query_id", pd.Series(dtype=str)).astype(str),
        checkpoint.get(ID_COLUMN, pd.Series(dtype=str)).astype(str),
    ))
    total = len(candidates)
    print(f"{reranker_name}: resume da {len(completed)}/{total}")

    for query_id in candidates["query_id"].drop_duplicates():
        qrows = candidates[candidates["query_id"].eq(query_id)].copy()
        qrows = qrows[
            [(str(r["query_id"]), str(r[ID_COLUMN])) not in completed for _, r in qrows.iterrows()]
        ]
        if qrows.empty:
            continue
        query_text = qrows["query"].iloc[0]

        for start in range(0, len(qrows), chunk_size):
            chunk = qrows.iloc[start:start+chunk_size]
            t0 = time.perf_counter()
            scores = reranker.score(query_text, chunk[DOCUMENT_COLUMN].astype(str).tolist())
            torch.cuda.synchronize()
            elapsed = time.perf_counter() - t0

            addition = pd.DataFrame({
                "query_id": chunk["query_id"].astype(str).to_numpy(),
                "query": chunk["query"].astype(str).to_numpy(),
                ID_COLUMN: chunk[ID_COLUMN].to_numpy(),
                "reranker": reranker_name,
                "reranker_score": scores,
            })
            checkpoint = addition.copy() if checkpoint.empty else pd.concat([checkpoint, addition], ignore_index=True)
            checkpoint = checkpoint.drop_duplicates(["query_id", ID_COLUMN], keep="last")
            atomic_write_csv(checkpoint[columns], checkpoint_path)
            completed.update(zip(addition["query_id"].astype(str), addition[ID_COLUMN].astype(str)))
            print(
                f"{reranker_name}: {len(completed)}/{total} salvate | "
                f"chunk={len(chunk)} in {elapsed:.1f}s ({elapsed/len(chunk):.3f}s/pair)"
            )

    return checkpoint[columns]


## Benchmark rapido su 20 coppie

Esegui questa cella prima del run completo. Il primo passaggio include anche parte dell'overhead di warm-up; il secondo è quello più rappresentativo.


In [ ]:
BENCHMARK_RERANKERS = [
    "bge_reranker_v2_m3",
    "qwen3_reranker_0.6b",
    "gte_multilingual_reranker",
]

sample = candidates.head(20)
query = sample["query"].iloc[0]
docs = sample[DOCUMENT_COLUMN].astype(str).tolist()

for name in BENCHMARK_RERANKERS:
    print("\nReranker:", name)
    model = RERANKER_FACTORIES[name]().load()
    # warm-up piccolo
    _ = model.score(query, docs[:2])
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = model.score(query, docs)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    print(f"20 coppie: {elapsed:.2f}s | {elapsed/20:.3f}s/coppia")
    model.unload()


## Scoring completo con checkpoint

Puoi eseguire un reranker alla volta. Se la sessione si interrompe, rilancia la cella: legge il CSV su Drive e riparte dalle coppie mancanti.


In [ ]:
RERANKERS_TO_RUN = [
    "bge_reranker_v2_m3",
    "qwen3_reranker_0.6b",
    "gte_multilingual_reranker",
]

for name in RERANKERS_TO_RUN:
    print("\n===", name, "===")
    model = RERANKER_FACTORIES[name]().load()
    checkpoint_path = SCORE_DIR / f"{name}.csv"
    score_incremental(
        candidates,
        model,
        name,
        checkpoint_path,
        chunk_size=CHECKPOINT_CHUNK_SIZE,
    )
    model.unload()

print("\nCheckpoint disponibili:")
for name in RERANKER_FACTORIES:
    path = SCORE_DIR / f"{name}.csv"
    if path.exists():
        frame = pd.read_csv(path)
        print(name, len(frame), "/", len(candidates), "->", path)
    else:
        print(name, "non iniziato")


## Rientro nel notebook 09

Copia i tre CSV prodotti in `artifacts/retrieval_ablation_2000_v1/reranker_scores/` del progetto locale, mantenendo i nomi:

- `bge_reranker_v2_m3.csv`
- `qwen3_reranker_0.6b.csv`
- `gte_multilingual_reranker.csv`

Poi riesegui soltanto la sezione 12 del notebook 09 con `RUN_RERANKER_SCORING = False`: il notebook ricostruirà `rerank@10`, `rerank@20` e `rerank@30` senza ulteriore inferenza.
